### =====
### Execution notes

Each device uses a different PART_ID with the same metadata and split settings.

Raw QA checkpoints are stored in `1_train_data/checkpoints`.
Filtered QA parts are stored in `1_train_data/filtered`.

Paths and checkpoint recovery were updated during repository cleanup.
### =====

In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "requirements-qa.txt").is_file():
        requirements_file = candidate / "requirements-qa.txt"
        break
else:
    raise FileNotFoundError("Could not find requirements-qa.txt")

%pip install -r "{requirements_file}"
# =====


In [20]:
import os
import re
import csv
import json
import time
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from huggingface_hub import hf_hub_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [ ]:
# =====
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "2_QA_GENERATION").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )

QA_DIR = PROJECT_ROOT / "2_QA_GENERATION"
TRAIN_DATA_DIR = QA_DIR / "1_train_data"

CHECKPOINT_DIR = TRAIN_DATA_DIR / "checkpoints"
FILTERED_DIR = TRAIN_DATA_DIR / "filtered"

for directory in [CHECKPOINT_DIR, FILTERED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
# =====

In [21]:
NUM_PARTS = 4

# CHANGE THIS ONLY ON EACH DEVICE:
PART_ID = 3

if PART_ID not in range(NUM_PARTS):
    raise ValueError(f"PART_ID must be between 0 and {NUM_PARTS - 1}")

RUN_NAME = f"train_part_{PART_ID}"

N_PAIRS_PER_TRANSCRIPT = 3
SAVE_EVERY = 10
SLEEP_SECONDS = 0
MAX_ROWS_IN_THIS_PART = None

print("Running:", RUN_NAME)


Running: train_part_3


In [ ]:
# =====
# REPO_ID = "KickItLikeShika/NileTTS-dataset"

# train_meta_path = hf_hub_download(
#     repo_id=REPO_ID,
#     repo_type="dataset",
#     filename="metadata_train.csv"
# )
train_meta_path = QA_DIR / "0_data" / "metadata_train.csv"

if not train_meta_path.is_file():
    raise FileNotFoundError(
        f"Missing training metadata: {train_meta_path}"
    )

# print(train_meta_path)
print("Train metadata:", train_meta_path)
# =====


/root/.cache/huggingface/hub/datasets--KickItLikeShika--NileTTS-dataset/snapshots/6905cb892bda2a315377178729ab2e718c0e63d4/metadata_train.csv


In [23]:
def load_niletts_metadata(path):
    df = pd.read_csv(
        path,
        sep="|",
        names=["audio_file", "text", "speaker_name"],
        dtype=str,
        engine="python",
        quoting=csv.QUOTE_NONE,
        on_bad_lines="skip"
    )

    if len(df) > 0 and str(df.iloc[0]["audio_file"]).strip() == "audio_file":
        df = df.iloc[1:].copy()

    for col in ["audio_file", "text", "speaker_name"]:
        df[col] = df[col].astype(str).str.strip()

    df = df.dropna(subset=["audio_file", "text", "speaker_name"])
    df = df[df["text"].str.len() > 0]
    df = df.drop_duplicates(subset=["audio_file"]).reset_index(drop=True)
    df["metadata_row_index"] = np.arange(len(df))
    return df


def extract_domain(audio_file):
    name = os.path.basename(str(audio_file)).lower()
    if name.startswith("sales"):
        return "sales"
    if name.startswith("medical"):
        return "medical"
    if name.startswith("general"):
        return "general"
    return "unknown"


train_df = load_niletts_metadata(train_meta_path)
train_df["domain"] = train_df["audio_file"].apply(extract_domain)

print("Official train metadata:", train_df.shape)
display(train_df.head())
display(train_df["domain"].value_counts())
display(train_df["speaker_name"].value_counts())


Official train metadata: (8571, 5)


,audio_file,text,speaker_name,metadata_row_index,domain
0,wavs/general_1_00041.wav,تماما وبجانب درجة الجودة العامة بيسألهم أسئلة ...,SPEAKER_02,0,general
1,wavs/medical_8_00009.wav,لأن العواقب أسوأ بكتير من اللي بنتخيله بص، الل...,SPEAKER_02,1,medical
2,wavs/sales_15_00036.wav,بنبص بره على العميل و القيمة اللي بيحصل عليها....,SPEAKER_01,2,sales
3,wavs/sales_3_00038.wav,تاني حاجة خططك كلها بتتدمر. حلم إنك تستقر في ب...,SPEAKER_02,3,sales
4,wavs/sales_26_00088.wav,هنا الأمور بدأت تتغير بشكل ملموس قوي، الإنترنت...,SPEAKER_02,4,sales


domain
sales      4478
general    2682
medical    1411
Name: count, dtype: int64

speaker_name
SPEAKER_01    4385
SPEAKER_02    4186
Name: count, dtype: int64

In [24]:
train_df["strata"] = train_df["domain"] + "_" + train_df["speaker_name"]

train_source, val_source = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df["strata"]
)

train_source = train_source.reset_index(drop=True)
val_source = val_source.reset_index(drop=True)

train_source["qa_train_source_index"] = np.arange(len(train_source))
val_source["qa_val_source_index"] = np.arange(len(val_source))

print("QA train source rows:", len(train_source))
print("QA val source rows:", len(val_source))


QA train source rows: 7713
QA val source rows: 858


In [25]:
all_indices = np.arange(len(train_source))
chunks = np.array_split(all_indices, NUM_PARTS)

part_indices = chunks[PART_ID]
part_source = train_source.iloc[part_indices].copy().reset_index(drop=True)

if MAX_ROWS_IN_THIS_PART is not None:
    part_source = part_source.head(MAX_ROWS_IN_THIS_PART).copy()

print("Part ID:", PART_ID)
print("Rows in this part:", len(part_source))
print("First qa_train_source_index:", part_source["qa_train_source_index"].min())
print("Last qa_train_source_index:", part_source["qa_train_source_index"].max())
display(part_source.head())


Part ID: 3
Rows in this part: 1928
First qa_train_source_index: 5785
Last qa_train_source_index: 7712


,audio_file,text,speaker_name,metadata_row_index,domain,strata,qa_train_source_index
0,wavs/sales_7_00047.wav,قصة ملهمة فعلاً. القصة دي بتجسد المبدأ اللي تم...,SPEAKER_01,6677,sales,sales_SPEAKER_01,5785
1,wavs/general_50_00039.wav,أولاً بتشيل المسئولية من على العروسة وأهلها ال...,SPEAKER_01,1102,general,general_SPEAKER_01,5786
2,wavs/sales_6_00064.wav,قرار الشراء المتسرع ده أكبر غلطة. الاستثمار ال...,SPEAKER_02,5515,sales,sales_SPEAKER_02,5787
3,wavs/sales_26_00116.wav,لأقصى درجة، لكن طبعاً تطبيق حاجة زي دي لسه قدا...,SPEAKER_02,83,sales,sales_SPEAKER_02,5788
4,wavs/sales_44_00065.wav,و كنقطة أخيرة كده للتفكير. بما إن المصادر ركزت...,SPEAKER_02,3898,sales,sales_SPEAKER_02,5789


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LOCAL_LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_NAME)

local_llm = AutoModelForCausalLM.from_pretrained(
    LOCAL_LLM_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

local_llm.eval()
print("Loaded:", LOCAL_LLM_NAME)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct


In [14]:
def normalize_spaces(text):
    return re.sub(r"\s+", " ", str(text)).strip()


def find_answer_start_soft(context, answer):
    context = str(context)
    answer = str(answer)

    exact = context.find(answer)
    if exact != -1:
        return exact

    context_norm = normalize_spaces(context)
    answer_norm = normalize_spaces(answer)

    norm_pos = context_norm.find(answer_norm)
    if norm_pos != -1:
        return norm_pos

    return -1


def build_qa_generation_prompt(context, domain, speaker_name, n_pairs=3):
    return (
        "You are creating a high-quality Arabic question-answering dataset from an Egyptian Arabic transcript.\n\n"
        f"Transcript domain: {domain}\n"
        f"Speaker: {speaker_name}\n\n"
        "Transcript:\n"
        f"{context}\n\n"
        f"Generate {n_pairs} useful QA pairs.\n\n"
        "Rules:\n"
        "1. Questions must be in Arabic.\n"
        "2. Questions must be answerable ONLY from the transcript.\n"
        "3. The answer must be a short span copied EXACTLY from the transcript.\n"
        "4. Do not write answers that are not literally present in the transcript.\n"
        "5. Avoid yes/no questions.\n"
        "6. Avoid vague questions such as what is mentioned.\n"
        "7. Use varied question types: main_idea, detail_extraction, intent, domain_specific, number_extraction.\n"
        "8. For medical transcripts, ask about symptoms, advice, treatment, diagnosis, or health information if available.\n"
        "9. For sales transcripts, ask about customer need, product/service, price, offer, request, or buying/selling information if available.\n"
        "10. For general transcripts, ask about the main idea, action, reason, or important detail.\n\n"
        "Return exactly this JSON structure and nothing else:\n"
        "{\n"
        '  "qa_pairs": [\n'
        "    {\n"
        '      "question": "Arabic question",\n'
        '      "answer": "exact answer span copied from transcript",\n'
        '      "question_type": "main_idea/detail_extraction/intent/domain_specific/number_extraction",\n'
        '      "difficulty": "easy/medium"\n'
        "    }\n"
        "  ]\n"
        "}\n"
    )


def extract_json_from_text(text):
    text = str(text).strip()

    if "```json" in text:
        text = text.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in text:
        text = text.split("```", 1)[1].split("```", 1)[0].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        text = text[start:end+1]

    return json.loads(text)


In [15]:
def call_llm_generate_qa(context, domain, speaker_name, n_pairs=3, max_retries=2):
    prompt = build_qa_generation_prompt(context, domain, speaker_name, n_pairs)

    system_prompt = (
        "You are an Arabic NLP dataset creator. "
        "Return JSON only. Do not return explanations. "
        "The answer must be copied exactly from the transcript."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]

    chat_text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        chat_text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(local_llm.device)

    last_error = None

    for attempt in range(max_retries):
        try:
            with torch.no_grad():
                outputs = local_llm.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=True,
                    temperature=0.2,
                    top_p=0.9,
                    pad_token_id=llm_tokenizer.eos_token_id
                )

            generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
            generated_text = llm_tokenizer.decode(generated_ids, skip_special_tokens=True)

            parsed = extract_json_from_text(generated_text)

            if isinstance(parsed, dict) and "qa_pairs" in parsed:
                return parsed["qa_pairs"]

            return []

        except Exception as e:
            last_error = e
            print(f"Local LLM parsing error, retrying: {e}")
            time.sleep(1)

    print("Failed local generation:", last_error)
    return []


In [ ]:
def generate_train_qa_checkpointed(
    source_df,
    checkpoint_path,
    processed_path,
    n_pairs_per_transcript=3,
    save_every=10,
    sleep_seconds=0
):
    df = source_df.copy().reset_index(drop=True)

    checkpoint_path = Path(checkpoint_path)
    processed_path = Path(processed_path)

    if checkpoint_path.exists():
        all_rows = pd.read_csv(checkpoint_path).to_dict("records")
        print(f"Loaded existing checkpoint: {checkpoint_path} with {len(all_rows)} QA rows")
    else:
        all_rows = []

    if processed_path.exists():
        processed_df = pd.read_csv(processed_path)
        processed_indices = set(processed_df["qa_train_source_index"].astype(int).tolist())
        print(f"Loaded processed log: {len(processed_indices)} transcript rows already processed")
    else:
    # ======
        # processed_indices = set()
        processed_indices = {
            int(row["source_row_index"])
            for row in all_rows
        }
        print(f"No processed log found; recovered {len(processed_indices)} source rows from the checkpoint")
    # =====

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"LLM QA generation: {RUN_NAME}"):
        qa_train_source_index = int(row["qa_train_source_index"])

        if qa_train_source_index in processed_indices:
            continue

        context_ref = str(row["text"]).strip()

        if len(context_ref.split()) < 5:
            print(f"Skipping short row {qa_train_source_index}")
            processed_indices.add(qa_train_source_index)
            continue

        qa_pairs = call_llm_generate_qa(
            context=context_ref,
            domain=row["domain"],
            speaker_name=row["speaker_name"],
            n_pairs=n_pairs_per_transcript
        )

        if not qa_pairs:
            print(f"No valid generation for qa_train_source_index {qa_train_source_index}. It will be retried next run.")
            continue

        added_for_this_row = 0

        for j, qa in enumerate(qa_pairs):
            if not isinstance(qa, dict):
                continue

            answer = normalize_spaces(qa.get("answer", ""))
            question = normalize_spaces(qa.get("question", ""))
            answer_start_ref = find_answer_start_soft(context_ref, answer)

            if not question or not answer:
                continue

            all_rows.append({
                "id": f"train_{qa_train_source_index}_{j}",
                "split": "train",
                "part_id": PART_ID,
                "metadata_row_index": int(row["metadata_row_index"]),
                "qa_train_source_index": qa_train_source_index,
                "audio_file": row["audio_file"],
                "speaker_name": row["speaker_name"],
                "domain": row["domain"],
                "context_ref": context_ref,
                "question": question,
                "answer": answer,
                "answer_start_ref": answer_start_ref,
                "valid_answer_span_ref": answer_start_ref != -1,
                "question_type": qa.get("question_type", ""),
                "difficulty": qa.get("difficulty", ""),
                "needs_manual_review": True
            })

            added_for_this_row += 1

        if added_for_this_row == 0:
            print(f"Generated output for qa_train_source_index {qa_train_source_index}, but no usable QA rows. It will be retried next run.")
            continue

        processed_indices.add(qa_train_source_index)

        if len(processed_indices) % save_every == 0:
            pd.DataFrame(all_rows).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

            # =====
            if not all_rows:
                raise RuntimeError(
                    "No QA pairs are available for this chunk. "
                    "Check generation errors and chunk contents before filtering."
                )
            # =====

            pd.DataFrame({"qa_train_source_index": sorted(list(processed_indices))}).to_csv(
                processed_path, index=False, encoding="utf-8-sig"
            )
            print(f"Saved checkpoint for {RUN_NAME}: {len(processed_indices)} transcript rows processed")

        time.sleep(sleep_seconds)

    pd.DataFrame(all_rows).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")
    pd.DataFrame({"qa_train_source_index": sorted(list(processed_indices))}).to_csv(
        processed_path, index=False, encoding="utf-8-sig"
    )

    print(f"Finished {RUN_NAME}: {len(processed_indices)} transcript rows processed")
    return pd.DataFrame(all_rows)


In [ ]:
# =====
# checkpoint_file = f"checkpoint_llm_qa_{RUN_NAME}.csv"
checkpoint_file = (CHECKPOINT_DIR / f"checkpoint_llm_qa_{RUN_NAME}.csv")

# processed_file = f"processed_llm_qa_{RUN_NAME}.csv"
processed_file = (CHECKPOINT_DIR / f"processed_llm_qa_{RUN_NAME}.csv")
# =====

qa_train_part_df = generate_train_qa_checkpointed(
    part_source,
    checkpoint_path=checkpoint_file,
    processed_path=processed_file,
    n_pairs_per_transcript=N_PAIRS_PER_TRANSCRIPT,
    save_every=SAVE_EVERY,
    sleep_seconds=SLEEP_SECONDS
)

print("Raw generated QA shape:", qa_train_part_df.shape)
display(qa_train_part_df.head())


Loaded existing checkpoint: checkpoint_llm_qa_train_part_3.csv with 445 QA rows
Loaded processed log: 150 transcript rows already processed


LLM QA generation: train_part_3:   0%|          | 0/1928 [00:00<?, ?it/s]

Saved checkpoint for train_part_3: 160 transcript rows processed
Saved checkpoint for train_part_3: 170 transcript rows processed
Saved checkpoint for train_part_3: 180 transcript rows processed
Saved checkpoint for train_part_3: 190 transcript rows processed
Saved checkpoint for train_part_3: 200 transcript rows processed
Saved checkpoint for train_part_3: 210 transcript rows processed
Local LLM parsing error, retrying: Invalid control character at: line 5 column 35 (char 146)
Saved checkpoint for train_part_3: 220 transcript rows processed
Saved checkpoint for train_part_3: 230 transcript rows processed
Saved checkpoint for train_part_3: 240 transcript rows processed
Saved checkpoint for train_part_3: 250 transcript rows processed
Saved checkpoint for train_part_3: 260 transcript rows processed
Saved checkpoint for train_part_3: 270 transcript rows processed
Saved checkpoint for train_part_3: 280 transcript rows processed
Saved checkpoint for train_part_3: 290 transcript rows process

,id,split,part_id,metadata_row_index,qa_train_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,valid_answer_span_ref,question_type,difficulty,needs_manual_review
0,train_5785_0,train,3,6677,5785,wavs/sales_7_00047.wav,SPEAKER_01,sales,قصة ملهمة فعلاً. القصة دي بتجسد المبدأ اللي تم...,ما هي القصة التي تذكرت؟,قصة ملهمة فعلاً.,0,True,domain_specific,easy,True
1,train_5785_1,train,3,6677,5785,wavs/sales_7_00047.wav,SPEAKER_01,sales,قصة ملهمة فعلاً. القصة دي بتجسد المبدأ اللي تم...,من الذي سيتمكن من تحقيق هذا المبدأ؟,ليه بيقدر يكمل و بيتعلم و هو بيكمل هو اللي في ...,-1,False,detail_extraction,medium,True
2,train_5785_2,train,3,6677,5785,wavs/sales_7_00047.wav,SPEAKER_01,sales,قصة ملهمة فعلاً. القصة دي بتجسد المبدأ اللي تم...,هل الفشل يعتبر نهاية العالم؟,الفشل مش عيب و مش نهاية العالم.,133,True,intent,easy,True
3,train_5786_0,train,3,1102,5786,wavs/general_50_00039.wav,SPEAKER_01,general,أولاً بتشيل المسئولية من على العروسة وأهلها ال...,المسؤولية من على العروسة وأهلها القرار؟,قرروا قرار الله بالحفاظ على كرامة الطرف المرفوض.,-1,False,domain_specific,easy,True
4,train_5786_1,train,3,1102,5786,wavs/general_50_00039.wav,SPEAKER_01,general,أولاً بتشيل المسئولية من على العروسة وأهلها ال...,ما هو النصيب الذي لا يُعْبُر عنه في هذه القضية؟,هذا ليس له أي قيمة بالنسبة لي.,-1,False,detail_extraction,medium,True


In [ ]:
def filter_valid_qa_for_train(df):
    before = len(df)
    df = df.copy()

    df["question"] = df["question"].astype(str).map(normalize_spaces)
    df["answer"] = df["answer"].astype(str).map(normalize_spaces)
    df["context_ref"] = df["context_ref"].astype(str).map(normalize_spaces)

    df = df[df["question"].str.len() > 5]
    df = df[df["answer"].str.len() > 0]
    df = df[df["valid_answer_span_ref"] == True]
    df = df.drop_duplicates(subset=["audio_file", "question", "answer"]).reset_index(drop=True)

    print(f"Kept {len(df)} / {before} QA rows after filtering")
    return df


qa_train_part_filtered_df = filter_valid_qa_for_train(qa_train_part_df)

# =====
# final_part_file = f"niletts_llm_qa_train_part_{PART_ID}.csv"
final_part_file = (FILTERED_DIR / f"niletts_llm_qa_train_part_{PART_ID}.csv")
# =====

qa_train_part_filtered_df.to_csv(final_part_file, index=False, encoding="utf-8-sig")

print("Saved:", final_part_file)
print("Final filtered part shape:", qa_train_part_filtered_df.shape)
display(qa_train_part_filtered_df.head())


Kept 3240 / 5756 QA rows after filtering
Saved: niletts_llm_qa_train_part_3.csv
Final filtered part shape: (3240, 16)


,id,split,part_id,metadata_row_index,qa_train_source_index,audio_file,speaker_name,domain,context_ref,question,answer,answer_start_ref,valid_answer_span_ref,question_type,difficulty,needs_manual_review
0,train_5785_0,train,3,6677,5785,wavs/sales_7_00047.wav,SPEAKER_01,sales,قصة ملهمة فعلاً. القصة دي بتجسد المبدأ اللي تم...,ما هي القصة التي تذكرت؟,قصة ملهمة فعلاً.,0,True,domain_specific,easy,True
1,train_5785_2,train,3,6677,5785,wavs/sales_7_00047.wav,SPEAKER_01,sales,قصة ملهمة فعلاً. القصة دي بتجسد المبدأ اللي تم...,هل الفشل يعتبر نهاية العالم؟,الفشل مش عيب و مش نهاية العالم.,133,True,intent,easy,True
2,train_5787_1,train,3,5515,5787,wavs/sales_6_00064.wav,SPEAKER_02,sales,قرار الشراء المتسرع ده أكبر غلطة. الاستثمار ال...,ما هو السؤال الذي أطرحه على المستمعين؟,قرار الشراء المتسرع ده أكبر غلطة.,0,True,main_idea,easy,True
3,train_5788_1,train,3,83,5788,wavs/sales_26_00116.wav,SPEAKER_02,sales,لأقصى درجة، لكن طبعاً تطبيق حاجة زي دي لسه قدا...,ما هي المشكلة الرئيسية التي تواجهنا عند ربط ال...,إزاي نربط العقود الذكية دي بالسجلات العقارية ا...,73,True,detail_extraction,medium,True
4,train_5788_2,train,3,83,5788,wavs/sales_26_00116.wav,SPEAKER_02,sales,لأقصى درجة، لكن طبعاً تطبيق حاجة زي دي لسه قدا...,هل هناك حاجة إلى تغيير وتحديث القوانين والتشري...,القوانين و التشريعات اللي محتاجة تتغير و تتطور...,206,True,intent,medium,True


In [29]:
print("Unique source transcript rows in this part:", qa_train_part_filtered_df["qa_train_source_index"].nunique())

print("Question type distribution:")
display(qa_train_part_filtered_df["question_type"].value_counts(dropna=False))

print("Difficulty distribution:")
display(qa_train_part_filtered_df["difficulty"].value_counts(dropna=False))

sample_cols = ["qa_train_source_index", "audio_file", "domain", "question", "answer", "context_ref"]

display(qa_train_part_filtered_df[sample_cols].sample(
    min(10, len(qa_train_part_filtered_df)),
    random_state=SEED
))


Unique source transcript rows in this part: 1510
Question type distribution:


question_type
domain_specific                                                         1082
detail_extraction                                                       1045
intent                                                                   575
main_idea                                                                283
main_idea/detail_extraction                                              101
number_extraction                                                         81
intent/main_idea                                                          31
main_idea/detail_extraction/intent/domain_specific/number_extraction      17
intent/domain_specific                                                     9
main_idea/detail_extraction/intent/domain_specific                         5
detail_extraction/intent/domain_specific                                   3
price                                                                      2
domain_specific/number_extraction                             

Difficulty distribution:


difficulty
easy      1982
medium    1255
hard         3
Name: count, dtype: int64

,qa_train_source_index,audio_file,domain,question,answer,context_ref
2330,7169,wavs/general_28_00023.wav,general,هل هي مكافأة حقيقية؟,مكافأة حقيقية؟,الحركة دي لوحدها أقوى أداة تفاوض طب نرجع تاني ...
1278,6541,wavs/sales_6_00040.wav,sales,كيف يمكنني التأكد من أن المطور هذا جيد وموثوق به؟,أعمل إيه بالزبط؟,و رابعاً لو حسيت بـ ضغط جديد عشان تمضي و تدفع ...
2519,7281,wavs/general_27_00048.wav,general,الكلمات الرئيسية التي تحدث عنها الجيل الحالي؟,كل جيل شايف إنه أحسن جيل,مزيكة الواقع بتاعهم بكل تعقيداته في النهاية ال...
321,5983,wavs/sales_6_00106.wav,sales,كيف يمكن للشرطي أن يتحقق من التزام البايع بالعقد؟,الضمانة بتاعتك لو البايع (سواء كان فرد أو شركة...,نجي لنصيحة قانونية مهمة (تكمل كلامنا عن العقود...
2841,7475,wavs/sales_28_00019.wav,sales,كيف يمكن للضحية أن تقول إن حظها شرير؟,أصل حظي وحش.,و إيه تحديدا اللي كان غلط أو ناقص و هو اللي سب...
1042,6402,wavs/sales_44_00034.wav,sales,ما هي الأهداف الرئيسية لعملية البيع؟,الهدف هو الفهم.,لأ. المصادر بتقترح إننا نبادر. نروح لناس نعتقد...
1338,6578,wavs/general_17_00002.wav,general,هل الرياضة ظهرت بشكل سريع وتحولت إلى عالم كامل؟,إزاي رياضة ظهرت شبه فجأة كده قدرت تخلق عالم كا...,إزاي رياضة ظهرت شبه فجأة كده قدرت تخلق عالم كا...
1210,6503,wavs/medical_18_00028.wav,medical,من هو المريض الذي يدرك أن هذه الأفكار غير منطقية؟,المريض نفسه,Sticky thoughts أفكار مش منطقية والمريض نفسه ع...
1313,6564,wavs/sales_42_00039.wav,sales,ما هو الهدف من كل شخص يجب أن يكون مستوعباً بشك...,الحقيقة اللي لازم الكل يبقى مستوعبها كويس,ان النجاح دا هيجي في كام شهر وخلاص. للأسف الشد...
2163,7069,wavs/general_56_00029.wav,general,كيف يمكن للعميل الحصول على استرجاع الأموال؟,مفيش استرجاع,مفيش ضمان، مفيش استرجاع، مفيش أي حق من حقوق ال...


In [ ]:
try:
    del local_llm
    del llm_tokenizer
except NameError:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA memory cleared.")
